In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.cleaning import replace_late_payment_sentinels

# index_col=0 must come before drop_duplicates: the unnamed ID column is
# unique per row, so leaving it in makes every row look distinct and dedup
# silently finds nothing.
df = pd.read_csv('cs-training.csv', index_col=0)

df = df.drop_duplicates()
df = df.reset_index(drop=True)

df = replace_late_payment_sentinels(df)



Sentinel occurrences per column (values 96/98):
  NumberOfTime30-59DaysPastDueNotWorse: 225
  NumberOfTimes90DaysLate: 225
  NumberOfTime60-89DaysPastDueNotWorse: 225


In [2]:
#late_cols = [c for c in df.columns if "PastDue" in c or "DaysLate" in c]

# value_counts on the tail shows the distribution's top end. Real counts
# taper off smoothly; a cluster sitting at 96/98 with nothing in between
# is the signature of a coded placeholder. 
#for c in late_cols:
    #print(c)
    #print(df[c].value_counts().sort_index().tail(10), "\n")

In [3]:
# since there's a cluster of 98 values and some 96, and has a large gap with other values this 
#is an indicator of a placeholder. based on previous test ran innthe previous cell. 
#next step is to turn the placeholrders to NaNs and NOT TO DROP THEM!



In [4]:
from src.modeling import split_data, build_pipeline, evaluate_cv

# The manual split that lived here moved into split_data() — same stratify=y,
# same seed, so the split itself is unchanged. From this point X_test / y_test
# are sealed: nothing fits on them, nothing scores on them, until the one
# final evaluation at the end of the project.
X_train, X_test, y_train, y_test = split_data(df)

print("train:", X_train.shape, "| test:", X_test.shape)
print(f"delinquency rate — train {y_train.mean():.4f} | test {y_test.mean():.4f}")

train: (119512, 11) | test: (29879, 11)
delinquency rate — train 0.0670 | test 0.0670


In [5]:
pipe = build_pipeline()
pipe  # last expression renders the pipeline, so every step and its params stay auditable

,steps,"[('impute', ...), ('scale', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,True
,keep_empty_features,False
,copy,True


In [6]:
# Cross-validated baseline on the TRAINING split only — the whole pipeline
# goes in, so imputer medians and scaler stats are re-learned inside every
# fold. X_test stays untouched.
scores = evaluate_cv(pipe, X_train, y_train)

per-fold AUC: [0.8214, 0.8211, 0.8197, 0.8187, 0.8208]
CV AUC      : 0.8203 ± 0.0010


In [7]:
# DebtRatio investigation — same method that exposed the 96/98 sentinels:
# look at the distribution's shape before changing anything. DebtRatio is
# debt/income, so it should sit near 0-1. Instead p75 = 0.9 and p90 = 1,274.
# A gap that size is a coding artifact, not a tail.
#
# Train only: what is learned here will drive a cleaning decision, so it must
# not be based on rows from the sealed test set.

d, inc = X_train.DebtRatio, X_train.MonthlyIncome

print("DebtRatio percentiles:")
print(d.describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string(), "\n")

# Suspected cause: with no usable denominator (income missing, 0, or 1) the
# field appears to hold a raw debt AMOUNT rather than a ratio.
income_unknown = inc.isna() | inc.le(1)
suspect = d.gt(1) & income_unknown      # ratio > 1 explained by unusable income
genuine = d.gt(1) & ~income_unknown     # ratio > 1 with a real denominator
normal = d.le(1)

print(f"median DebtRatio, income known   : {d[~income_unknown].median():>10,.2f}")
print(f"median DebtRatio, income unknown : {d[income_unknown].median():>10,.0f}")
print("  -> the second reads as monthly dollars, not a proportion\n")

# The decisive test is risk, not magnitude: if both groups were the same
# quantity, they would carry the same default rate.
print("default rate by group:")
for label, m in [("DebtRatio <= 1               ", normal),
                 ("DebtRatio > 1, real income   ", genuine),
                 ("DebtRatio > 1, income NaN/0/1", suspect)]:
    print(f"  {label}  n={m.sum():>7,}   {y_train[m].mean():.4f}")
print(f"  {'overall base rate':<29}  n={len(y_train):>7,}   {y_train.mean():.4f}")

DebtRatio percentiles:
count    119512.0
mean        357.6
std        2213.1
min           0.0
50%           0.4
75%           0.9
90%        1274.0
95%        2460.0
99%        4937.9
max      329664.0 

median DebtRatio, income known   :       0.29
median DebtRatio, income unknown :      1,162
  -> the second reads as monthly dollars, not a proportion

default rate by group:
  DebtRatio <= 1                 n= 91,436   0.0681
  DebtRatio > 1, real income     n=  4,054   0.1260
  DebtRatio > 1, income NaN/0/1  n= 24,022   0.0527
  overall base rate              n=119,512   0.0670


In [8]:
# ---- Data repairs, round 2 --------------------------------------------------
# The investigation above showed DebtRatio holding two units in one column:
# ratios when income is known, raw dollar amounts when it is not. The repair
# (NaN + a debt_ratio_artifact flag) lives in src/cleaning.py beside the
# sentinel fix, and load_gmsc() now applies every repair in one call so the
# training and Kaggle test files can never drift apart in cleaning.
# The rule was derived from X_train diagnostics only — the sealed test rows
# never influenced it.
from src.cleaning import load_gmsc

df = load_gmsc("cs-training.csv")

# Same row set and same seed, so this re-split reproduces the identical
# train/test partition — the seal on X_test carries over.
X_train, X_test, y_train, y_test = split_data(df)
print("\ntrain:", X_train.shape, "| test:", X_test.shape)

Sentinel occurrences per column (values 96/98):
  NumberOfTime30-59DaysPastDueNotWorse: 225
  NumberOfTimes90DaysLate: 225
  NumberOfTime60-89DaysPastDueNotWorse: 225
DebtRatio artifact rows (income missing/0/1): 31433
  of which DebtRatio > 1: 29976
  genuine ratios > 1 kept: 5154
Impossible ages (<18): 1

train: (119512, 12) | test: (29879, 12)


In [9]:
# Linear baseline on the repaired data. The repair alone barely moves the
# number — the scaler is still fighting untransformed heavy tails — which is
# itself informative: the artifacts corrupted meaning more than they
# corrupted this particular score.
scores_repaired = evaluate_cv(build_pipeline(), X_train, y_train)

per-fold AUC: [0.8219, 0.8217, 0.8195, 0.8192, 0.8218]
CV AUC      : 0.8208 ± 0.0012


In [10]:
# log1p re-expression of the heavy-tailed columns (src/features.py). It is
# parameter-free — nothing is learned from the data — so applying it outside
# the CV folds cannot leak; only FITTED parameters leak. This is the linear
# model's binding constraint: one coefficient per column cannot survive a
# feature whose max is five orders of magnitude above its median.
from src.features import engineer_features

X_train_fe = engineer_features(X_train)
scores_fe = evaluate_cv(build_pipeline(), X_train_fe, y_train)

per-fold AUC: [0.8531, 0.8574, 0.8545, 0.8576, 0.8583]
CV AUC      : 0.8562 ± 0.0020


In [11]:
# Model escalation: histogram gradient boosting (src/modeling.py). Raw
# repaired features go in — no imputer, no scaler, no log transforms: the
# estimator routes NaN natively (missingness becomes a split direction,
# which is exactly right for data whose NaNs are deliberate "unknown"
# markers), and threshold splits are indifferent to scale and to monotone
# re-expressions. Config chosen by a CV sweep on the training split.
from src.modeling import build_gbm_pipeline

scores_gbm = evaluate_cv(build_gbm_pipeline(), X_train, y_train)

per-fold AUC: [0.8638, 0.8648, 0.8632, 0.8685, 0.8644]
CV AUC      : 0.8649 ± 0.0019


In [12]:
# ---- FINAL EVALUATION: the test set is spent here, once ---------------------
# Model selection is over (the GBM won on CV), so exactly one model gets to
# see X_test, exactly once. If the number lands within ~2 fold-stds of the
# CV mean, the CV protocol was honest; either way, nothing is tuned after
# this cell — reacting to a test score turns it into one more validation set.
from src.modeling import evaluate_holdout

final_model = build_gbm_pipeline()
final_model.fit(X_train, y_train)
auc_test = evaluate_holdout(final_model, X_test, y_test, cv_reference=scores_gbm)

HELD-OUT TEST AUC: 0.8642   (this number is now spent)
CV estimate was  : 0.8649 ± 0.0019   (gap -0.0007 = -0.4 fold-stds)


In [13]:
# ---- Kaggle submission ------------------------------------------------------
# Refit on ALL labeled rows: selection is finished and the holdout has served
# its purpose, so holding 20% back now would only starve the shipped model.
# The Kaggle file goes through the exact same load_gmsc() path as training
# (minus dedup — every Id must receive a prediction, and the file's index IS
# the Id column the submission format requires).
X_full = df.drop(columns=["SeriousDlqin2yrs"])
y_full = df["SeriousDlqin2yrs"]

deploy_model = build_gbm_pipeline()
deploy_model.fit(X_full, y_full)

kaggle = load_gmsc("cs-test.csv", training=False)
X_kaggle = kaggle.drop(columns=["SeriousDlqin2yrs"])  # empty in the test file

submission = pd.DataFrame(
    {"Id": X_kaggle.index, "Probability": deploy_model.predict_proba(X_kaggle)[:, 1]}
)
submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv:", submission.shape, "| Id", submission.Id.min(), "->", submission.Id.max())
print(submission.head(3).to_string(index=False))

Sentinel occurrences per column (values 96/98):
  NumberOfTime30-59DaysPastDueNotWorse: 214
  NumberOfTimes90DaysLate: 214
  NumberOfTime60-89DaysPastDueNotWorse: 214
DebtRatio artifact rows (income missing/0/1): 21530
  of which DebtRatio > 1: 20176
  genuine ratios > 1 kept: 3402
Impossible ages (<18): 0

submission.csv: (101503, 2) | Id 1 -> 101503
 Id  Probability
  1     0.079709
  2     0.047281
  3     0.020185


In [14]:
# ---- Project results --------------------------------------------------------
# Computed from the live variables above so this summary can never go stale.
print("logistic, repaired data      CV AUC "
      f"{scores_repaired.mean():.4f} ± {scores_repaired.std():.4f}")
print("logistic, + log1p tails      CV AUC "
      f"{scores_fe.mean():.4f} ± {scores_fe.std():.4f}")
print("gradient boosting (final)    CV AUC "
      f"{scores_gbm.mean():.4f} ± {scores_gbm.std():.4f}")
print(f"final model, sealed test        AUC {auc_test:.4f}   (one-shot)")
print()
print("(pre-repair logistic baseline was 0.8203 ± 0.0010;")


logistic, repaired data      CV AUC 0.8208 ± 0.0012
logistic, + log1p tails      CV AUC 0.8562 ± 0.0020
gradient boosting (final)    CV AUC 0.8649 ± 0.0019
final model, sealed test        AUC 0.8642   (one-shot)

(pre-repair logistic baseline was 0.8203 ± 0.0010;
